# Mangrove Pulse — conditional drone resource audit with SymPy

This small companion checks general algebra in our existing hypothetical supported-sampling model. It needs no MATLAB licence, satellite files, API keys, or paid runtime. Use a free CPU runtime. Stored outputs will show what was actually executed.

The implemented PoC is a frozen Sentinel-2/GMW reference-agreement analysis at Khor Kalba, with review cases C001, C017 and C027. The amphibious drone is **concept-only**. This notebook supplies parameter-dependent design/rejection conditions for a future task, not independent 2018 cover labels, hardware superiority, or ecological benefit.

C001/C017 ask photographic interpretation questions that simpler methods may answer. C027 may motivate a separate current contact-sensing question if a practitioner justifies it. No case cluster is a sampling station or approved flight/landing route. Today’s observations cannot directly validate 2018 cover.


## Model and units

Let baseline airborne power be $P>0$, extra airborne penalty $D\geq0$, supported power $W>0$, travel time $T\geq0$, attempted sites $N\in\mathbb{Z}_{>0}$, equal dwell $d\geq0$, extra airborne transitions $\tau\geq0$, extra supported overhead $O\geq0$ Wh, and usable battery $B>0$ Wh.

Scaled energies in watt-seconds are:

$$H=P(T+Nd),\qquad S=(P+D)(T+N\tau)+WNd+3600O.$$

These are the saved model's energy accounting, not full physics. Its baseline uses ideal induced hover power with a hypothetical loss multiplier; winds, corrosion, transition dynamics, access, recovery, quality and safety are not established. $P+D$ assumes the original equal rotor-area/common-load model and nonnegative added burden.

Define $K=DT+(P+D)N\tau+3600O$, $A=3600B-(P+D)(T+N\tau)-3600O$. Then $H-S=N(P-W)d-K$. With $P>W$, strict energy advantage needs $d>K/[N(P-W)]$. With $W\geq P$, nonnegative penalties rule out strict energy advantage. Both platforms fitting the same battery also needs the two upper bounds shown in the results.

Accepted fractions $0<p_H,p_S\leq1$ are unknown. Comparing **expected totals per expected accepted record**, supported energy improves only when $p_S/p_H>S/H$. This does not assume measured quality parity. It is not an expectation of a per-mission ratio, which may be undefined when no records are accepted.

Accounted duration is a subtotal: $T_H=T+Nd$, $T_S=T+Nd+N\tau$. Positive transition time and equal acceptance exclude a time advantage in this accounting; unspecified overhead duration is not silently zero.


In [1]:
import importlib.util, subprocess, sys
if importlib.util.find_spec("sympy") is None:
    subprocess.check_call([sys.executable,"-m","pip","install","sympy==1.14.0"])
import sympy as sp
print("SymPy:", sp.__version__)
print("Free CPU only; no MATLAB, API key, satellite upload or model fitting.")


SymPy: 1.14.0
Free CPU only; no MATLAB, API key, satellite upload or model fitting.


In [2]:
import sympy as sp
import json, platform
from datetime import datetime, timezone

# General symbols; no craft measurements or field observations are supplied.
P, W, B = sp.symbols("P W B", positive=True)
D, T, d, tau, O, u = sp.symbols("D T d tau O u", nonnegative=True)
N = sp.symbols("N", positive=True, integer=True)
pH, pS = sp.symbols("pH pS", positive=True)  # additionally <= 1 for accepted fractions
H = P * (T + N*d)
S = (P+D) * (T + N*tau) + W*N*d + 3600*O
K = D*T + (P+D)*N*tau + 3600*O
L = N*(P-W)
A = 3600*B - (P+D)*(T+N*tau) - 3600*O
TH, TS = T+N*d, T+N*d+N*tau

residuals = {
    "M1_energy_difference": H-S-(L*d-K),
    "M2_no_advantage_when_W_ge_P": (H-S).subs(W,P+u)-(-K-N*d*u),
    "M3_energy_per_expected_accepted_record": H*pS-S*pH-((H-S)*pH-H*(pH-pS)),
    "M4_accounted_time_penalty": TS-TH-N*tau,
    "M5_supported_battery_budget": 3600*B-S-(A-N*W*d),
    "M6_supported_window_cross_product": A*N*(P-W)-N*W*K-N*(A*(P-W)-W*K),
    "M7_hover_battery_budget": 3600*B-H-(3600*B-P*T-N*P*d),
}
checks = {name: str(sp.expand(value)) for name,value in residuals.items()}
assert all(value == "0" for value in checks.values()), checks
lower = K/(N*(P-W))
upper_supported = A/(N*W)
upper_hover = (3600*B-P*T)/(N*P)
assert sp.factor(upper_supported-lower) == sp.factor((A*(P-W)-W*K)/(N*W*(P-W)))
assert sp.factor(upper_hover-lower) == sp.factor(((3600*B-P*T)*(P-W)-P*K)/(N*P*(P-W)))

report = {
    "status":"succeeded",
    "scope":"Exact symbolic identities and conditional resource screens; not a measured drone experiment or a formal proof-assistant verification.",
    "executed_utc":datetime.now(timezone.utc).isoformat(),
    "sympy_version":sp.__version__,
    "python_version":platform.python_version(),
    "source_model_sha256":"f30477579ba27d620219957e42d646beeb08093b53510ba3def0009acf1bca68",
    "symbol_domains": {
        "P_W_B":"strictly positive; watts, watts, usable watt-hours",
        "D_T_d_tau_O":"nonnegative; watts, seconds, seconds/site, seconds/site, watt-hours",
        "N":"positive integer attempted sites",
        "pH_pS":"0 < accepted fraction <= 1, unknown",
        "energy_units":"H and S are watt-seconds, divide by 3600 for Wh",
    },
    "zero_residuals":checks,
    "additional_rational_identities_passed":2,
    "conditions":{
        "strict_energy_advantage":"If P>W, d>K/[N(P-W)]. If W>=P, no strict advantage under nonnegative penalties.",
        "energy_per_expected_accepted_record":"If H,S>0, advantage iff pS/pH>S/H. Ratio of expected totals over equal attempted tasks, not expected per-mission ratio.",
        "time_per_expected_accepted_record":"If TH,TS>0, advantage iff pS/pH>TS/TH; positive tau and equal acceptance rule out lower accounted time.",
        "supported_battery_screen":"d<=A/(NW), and d>=0. A negative upper bound means no feasible nonnegative dwell.",
        "supported_advantage_window":"If P>W, nonempty strict-energy-advantage plus supported-battery interval iff A(P-W)>WK.",
        "both_platforms_within_same_B":"If P>W, require K/[N(P-W)]<min(A/[NW],(3600B-PT)/[NP]); equivalently both positive cross-product inequalities.",
        "hover_window_cross_product":"(3600B-PT)(P-W)>PK",
    },
    "claims_not_established":[
        "independent 2018 Khor Kalba cover accuracy",
        "measured power, endurance, accepted observations or amphibious transitions",
        "superiority over a UAE drone or simpler observation method",
        "sensor calibration, safety, permissions, habitat disturbance",
        "planting suitability, survival or carbon benefit",
    ],
}
print("PASS: 7 zero-residual identities + 2 rational identities")
print("Model-only finding: added transition time cannot improve equal-acceptance accounted duration.")
print("Model-only finding: energy benefit requires sufficient dwell AND battery AND accepted-record gates.")
print(json.dumps(report,indent=2))


PASS: 7 zero-residual identities + 2 rational identities
Model-only finding: added transition time cannot improve equal-acceptance accounted duration.
Model-only finding: energy benefit requires sufficient dwell AND battery AND accepted-record gates.
{
  "status": "succeeded",
  "scope": "Exact symbolic identities and conditional resource screens; not a measured drone experiment or a formal proof-assistant verification.",
  "executed_utc": "2026-10-08T10:04:54.855043+00:00",
  "sympy_version": "1.14.0",
  "python_version": "3.13.16",
  "source_model_sha256": "f30477579ba27d620219957e42d646beeb08093b53510ba3def0009acf1bca68",
  "symbol_domains": {
    "P_W_B": "strictly positive; watts, watts, usable watt-hours",
    "D_T_d_tau_O": "nonnegative; watts, seconds, seconds/site, seconds/site, watt-hours",
    "N": "positive integer attempted sites",
    "pH_pS": "0 < accepted fraction <= 1, unknown",
    "energy_units": "H and S are watt-seconds, divide by 3600 for Wh"
  },
  "zero_residual

## What the calculation can and cannot establish

Exact zero residuals verify seven identities; two additional rational identities check the interval conditions. The inequality directions also require the stated positive denominators. SymPy simplification is not a formal proof-assistant certificate and is not an observation.

These conditions make the concept falsifiable: reject energy advantage at insufficient dwell, excessive support burden or inadequate acceptance; reject a mission if either required battery screen fails. No numerical inputs or measurements are invented. No 2018 thresholds, split, predictions or reference results are changed.

To claim real superiority later, compare the same justified observation task with ordinary drone/shore/boat methods, independently assess accepted records, and measure complete energy, elapsed time, failed attempts, recovery and disturbance. To claim ecological benefit, suitability and replicated baseline/follow-up outcomes are separate evidence gates.

### Sources and reproducibility

The model is transcribed from the preserved `sampling_energy_model.py::evaluate` function; SHA-256 is recorded in the output. Obtain that model with the team's submission repository, not through an unverified external copy. This notebook is self-contained for algebra and does not reload satellite data.

[SymPy project](https://github.com/sympy/sympy) · [SymPy simplification documentation](https://docs.sympy.org/latest/tutorials/intro-tutorial/simplification.html)

SymPy is BSD-licensed. Use the installed Colab version and record it; if missing, install 1.14.0 once. The same exact elementary operations can be rerun with `sympy==1.14.0`. Saving in Drive does not by itself grant judges access; the team manages sharing separately.


In [3]:
from pathlib import Path
import tempfile, hashlib, zipfile
out = Path(tempfile.mkdtemp(prefix="mangrove_pulse_sympy_"))
json_path = out / "symbolic_resource_audit.json"
json_path.write_text(json.dumps(report,indent=2)+"\n",encoding="utf-8")
readme = out / "README.txt"
readme.write_text("Mangrove Pulse: conditional symbolic resource audit.\n"
                  "PASS means algebra matched under documented assumptions, NOT hardware or ecological proof.\n"
                  "Seven zero-residual identities and two rational identities; no new datasets.\n"
                  "Preserve this JSON together with the executed notebook.\n",encoding="utf-8")
archive = out / "Mangrove_Pulse_SymPy_Results.zip"
with zipfile.ZipFile(archive,"w",zipfile.ZIP_DEFLATED) as z:
    z.write(json_path,json_path.name)
    z.write(readme,readme.name)
print("Status:", report["status"])
print("ZIP bytes:", archive.stat().st_size)
print("ZIP SHA-256:", hashlib.sha256(archive.read_bytes()).hexdigest())
from google.colab import files
files.download(str(archive))


Status: succeeded
ZIP bytes: 1596
ZIP SHA-256: f4d123f339db0ef874d5a45a85ee4e00cfea4bd97bbf204ffd8620661c5fe004


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>